In [6]:
# ============================================================
# Geo-FNO training on elasticity dataset (geometry -> sigma)
# - Input per sample: deformed coordinates (x, y) on a 2D grid
# - Output per sample: stress field sigma on that same geometry
# - Uses Geo-FNO (FNO2d + IPHI) with mesh-free spectral operators
# ============================================================

import os
import json
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim import Adam

from geo_FNO_def import FNO2d, IPHI


class ElasticityGeoFNODataset(Dataset):
    """
    Expects arrays shaped (nx, ny, nsamples):
      X, Y, sigma

    Returns one sample as:
      pos   : (N,2) flattened coordinates
      u_in  : (N,2) input features = geometry coordinates (x,y)
      u_out : (N,1) target stress sigma
    """
    def __init__(self, x_path: str, y_path: str, sigma_path: str):
        super().__init__()
        self.X = np.load(x_path)
        self.Y = np.load(y_path)
        self.S = np.load(sigma_path)

        assert self.X.shape == self.Y.shape == self.S.shape, (
            f"Shape mismatch: X={self.X.shape}, Y={self.Y.shape}, S={self.S.shape}"
        )
        self.nx, self.ny, self.nsamples = self.X.shape

    def __len__(self):
        return self.nsamples

    def __getitem__(self, idx: int):
        x2 = self.X[:, :, idx].astype(np.float32)
        y2 = self.Y[:, :, idx].astype(np.float32)
        s2 = self.S[:, :, idx].astype(np.float32)

        pos = np.stack([x2, y2], axis=-1).reshape(-1, 2)      # (N,2)
        u_in = pos.copy()                                      # (N,2)
        u_out = s2.reshape(-1, 1)                              # (N,1)

        code = np.zeros((42,), dtype=np.float32)
        code[0] = 0.5
        code[1] = 0.5

        return (
            torch.from_numpy(pos),
            torch.from_numpy(u_in),
            torch.from_numpy(u_out),
            torch.from_numpy(code),
        )


def collate_bs1(batch):
    return batch[0]


def random_subset_indices(total: int, n: int, seed: int):
    n = min(n, total)
    rng = np.random.default_rng(seed)
    return rng.choice(total, size=n, replace=False).tolist()


def relative_rmse(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    # Dimensionless RMS error: 1e-2 corresponds to about 1% RMS error.
    mse = torch.mean((pred - target) ** 2)
    ref = torch.mean(target ** 2)
    return torch.sqrt(mse / torch.clamp(ref, min=eps))


def compute_io_stats_from_indices(ds, indices, in_channels: int, out_channels: int):
    sum_in = torch.zeros(in_channels, dtype=torch.float64)
    sq_in = torch.zeros(in_channels, dtype=torch.float64)
    n_in = 0

    sum_out = torch.zeros(out_channels, dtype=torch.float64)
    sq_out = torch.zeros(out_channels, dtype=torch.float64)
    n_out = 0

    for i in indices:
        _, u_in, u_out, _ = ds[i]
        u_in64 = u_in.to(torch.float64)
        u_out64 = u_out.to(torch.float64)

        sum_in += u_in64.sum(dim=0)
        sq_in += (u_in64 * u_in64).sum(dim=0)
        n_in += int(u_in64.shape[0])

        sum_out += u_out64.sum(dim=0)
        sq_out += (u_out64 * u_out64).sum(dim=0)
        n_out += int(u_out64.shape[0])

    mean_in = sum_in / max(1, n_in)
    mean_out = sum_out / max(1, n_out)

    var_in = torch.clamp(sq_in / max(1, n_in) - mean_in * mean_in, min=1e-12)
    var_out = torch.clamp(sq_out / max(1, n_out) - mean_out * mean_out, min=1e-12)

    return {
        "u_in_mean": mean_in.to(torch.float32),
        "u_in_std": torch.sqrt(var_in).to(torch.float32),
        "u_out_mean": mean_out.to(torch.float32),
        "u_out_std": torch.sqrt(var_out).to(torch.float32),
    }


def train_geofno_elasticity(
    x_path: str,
    y_path: str,
    sigma_path: str,
    ntrain: int = 1600,
    ntest: int = 400,
    split_seed: int = 0,
    random_split: bool = True,
    batch_size: int = 1,
    epochs: int = 500,
    learning_rate_fno: float = 1e-3,
    learning_rate_iphi: float = 1e-4,
    modes: int = 20,
    width: int = 32,
    s1: int = 65,
    s2: int = 41,
    device: str = "cuda:0",
    val_patience: int = 25,
    improve_thresh: float = 1e-3,
    use_io_normalization: bool = True,
):
    ds = ElasticityGeoFNODataset(x_path, y_path, sigma_path)
    total = len(ds)
    print(f"Loaded elasticity dataset with {total} samples; grid=({ds.nx},{ds.ny})")

    if random_split:
        all_idx = np.arange(total)
        rng = np.random.default_rng(split_seed)
        rng.shuffle(all_idx)
        ntrain_eff = min(ntrain, total)
        ntest_eff = min(ntest, total - ntrain_eff)
        train_idx = all_idx[:ntrain_eff].tolist()
        test_idx = all_idx[ntrain_eff:ntrain_eff + ntest_eff].tolist()
    else:
        ntrain_eff = min(ntrain, total)
        ntest_eff = min(ntest, total - ntrain_eff)
        train_idx = list(range(ntrain_eff))
        test_idx = list(range(ntrain_eff, ntrain_eff + ntest_eff))

    print("Train subset size:", len(train_idx))
    print("Test subset size:", len(test_idx))

    train_loader = DataLoader(
        Subset(ds, train_idx),
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_bs1,
    )
    test_loader = DataLoader(
        Subset(ds, test_idx),
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_bs1,
    )

    # L from physical domain extents in the first sample
    L1 = float(ds.X[:, :, 0].max() - ds.X[:, :, 0].min())
    L2 = float(ds.Y[:, :, 0].max() - ds.Y[:, :, 0].min())
    print("Using L:", [L1, L2])

    model = FNO2d(
        modes, modes, width,
        in_channels=2, out_channels=1,
        is_mesh=False, s1=s1, s2=s2,
        L=[L1, L2],
    ).to(device)
    model_iphi = IPHI(width=32, device=device).to(device)

    if use_io_normalization:
        stats = compute_io_stats_from_indices(ds, train_idx, in_channels=2, out_channels=1)
        model.set_io_normalization(
            stats["u_in_mean"], stats["u_in_std"],
            stats["u_out_mean"], stats["u_out_std"],
            enabled=True,
        )
        print("I/O normalization enabled")
        print("  u_in mean/std:", stats["u_in_mean"].tolist(), stats["u_in_std"].tolist())
        print("  u_out mean/std:", stats["u_out_mean"].tolist(), stats["u_out_std"].tolist())
    else:
        model.set_io_normalization_enabled(False)
        print("I/O normalization disabled")

    opt_fno = Adam(model.parameters(), lr=learning_rate_fno, weight_decay=1e-4)
    opt_iphi = Adam(model_iphi.parameters(), lr=learning_rate_iphi, weight_decay=1e-4)
    sched_fno = torch.optim.lr_scheduler.CosineAnnealingLR(opt_fno, T_max=epochs)
    sched_iphi = torch.optim.lr_scheduler.CosineAnnealingLR(opt_iphi, T_max=epochs)
    loss_fn = nn.MSELoss()

    best_val_rel = 1e12
    epochs_no_improve = 0
    best_state_model = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    best_state_iphi = {k: v.detach().cpu().clone() for k, v in model_iphi.state_dict().items()}

    for ep in range(epochs):
        model.train()
        model_iphi.train()
        train_mse = 0.0
        train_rel = 0.0

        for pos, u_in, u_out, code42 in train_loader:
            pos = pos.unsqueeze(0).to(device)       # (1,N,2)
            u_in = u_in.unsqueeze(0).to(device)     # (1,N,2)
            u_out = u_out.unsqueeze(0).to(device)   # (1,N,1)
            code42 = code42.unsqueeze(0).to(device) # (1,42)

            opt_fno.zero_grad()
            opt_iphi.zero_grad()

            pred = model(u_in, code=code42, x_in=pos, x_out=pos, iphi=model_iphi)  # (1,N,1)
            mse = loss_fn(pred, u_out)
            mse.backward()

            opt_fno.step()
            opt_iphi.step()

            train_mse += float(mse.item())
            train_rel += float(relative_rmse(pred.detach(), u_out).item())

        train_mse /= max(1, len(train_loader))
        train_rel /= max(1, len(train_loader))

        model.eval()
        model_iphi.eval()
        val_mse = 0.0
        val_rel = 0.0
        with torch.no_grad():
            for pos, u_in, u_out, code42 in test_loader:
                pos = pos.unsqueeze(0).to(device)
                u_in = u_in.unsqueeze(0).to(device)
                u_out = u_out.unsqueeze(0).to(device)
                code42 = code42.unsqueeze(0).to(device)

                pred = model(u_in, code=code42, x_in=pos, x_out=pos, iphi=model_iphi)
                val_mse += float(loss_fn(pred, u_out).item())
                val_rel += float(relative_rmse(pred, u_out).item())

        val_mse /= max(1, len(test_loader))
        val_rel /= max(1, len(test_loader))
        sched_fno.step()
        sched_iphi.step()

        print(
            f"ep={ep:04d} "
            f"train_relRMSE={train_rel:.6e} | "
            f"val_relRMSE={val_rel:.6e} | "
            f"train_MSE={train_mse:.6e} | val_MSE={val_mse:.6e}"
        )

        if val_rel < best_val_rel * (1 - improve_thresh):
            best_val_rel = val_rel
            best_state_model = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_state_iphi = {k: v.detach().cpu().clone() for k, v in model_iphi.state_dict().items()}
            epochs_no_improve = 0
        else:
            if best_val_rel * (1 - improve_thresh) < val_rel < best_val_rel:
                best_val_rel = val_rel
                best_state_model = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                best_state_iphi = {k: v.detach().cpu().clone() for k, v in model_iphi.state_dict().items()}
            epochs_no_improve += 1
            if epochs_no_improve >= val_patience:
                print("Early stop triggered.")
                break

        if ep % 100 == 0:
            os.makedirs("./checkpoints", exist_ok=True)
            torch.save(model.state_dict(), f"./checkpoints/geofno_elasticity_ep{ep}.pt")
            torch.save(model_iphi.state_dict(), f"./checkpoints/iphi_elasticity_ep{ep}.pt")

    model.load_state_dict(best_state_model)
    model_iphi.load_state_dict(best_state_iphi)
    print(f"Best val relRMSE: {best_val_rel:.6e} ({100.0 * best_val_rel:.3f}%)")

    return model, model_iphi


In [7]:
x_path = "/scratch/mnhagen/datasets/elasticity/Random_UnitCell_Deform_X_10_interp.npy"
y_path = "/scratch/mnhagen/datasets/elasticity/Random_UnitCell_Deform_Y_10_interp.npy"
sigma_path = "/scratch/mnhagen/datasets/elasticity/Random_UnitCell_Deform_sigma_10_interp.npy"

device = "cuda:0" if torch.cuda.is_available() else "cpu"

model, model_iphi = train_geofno_elasticity(
    x_path=x_path,
    y_path=y_path,
    sigma_path=sigma_path,
    ntrain=1600,
    ntest=400,
    split_seed=0,
    random_split=True,
    batch_size=1,
    epochs=500,
    learning_rate_fno=1e-3,
    learning_rate_iphi=1e-4,
    modes=20,
    width=32,
    s1=65,
    s2=41,
    device=device,
    val_patience=25,
    use_io_normalization=True,
)


Loaded elasticity dataset with 2000 samples; grid=(65,41)
Train subset size: 1600
Test subset size: 400
Using L: [1.0, 1.0]
I/O normalization enabled
  u_in mean/std: [0.5059542059898376, 0.5001823306083679] [0.3129917085170746, 0.3087625801563263]
  u_out mean/std: [191.20799255371094] [137.00364685058594]
ep=0000 train_relRMSE=1.862314e-01 | val_relRMSE=1.291702e-01 | train_MSE=3.257420e+03 | val_MSE=9.099256e+02
ep=0001 train_relRMSE=1.176954e-01 | val_relRMSE=1.041513e-01 | train_MSE=7.998399e+02 | val_MSE=6.134496e+02
ep=0002 train_relRMSE=1.015601e-01 | val_relRMSE=8.604943e-02 | train_MSE=6.021124e+02 | val_MSE=4.189180e+02
ep=0003 train_relRMSE=1.035983e-01 | val_relRMSE=1.317836e-01 | train_MSE=6.366293e+02 | val_MSE=9.129221e+02
ep=0004 train_relRMSE=8.753056e-02 | val_relRMSE=6.712917e-02 | train_MSE=4.467395e+02 | val_MSE=2.534415e+02
ep=0005 train_relRMSE=7.884508e-02 | val_relRMSE=6.993713e-02 | train_MSE=3.666244e+02 | val_MSE=2.723861e+02
ep=0006 train_relRMSE=7.441290e

In [ ]:
import os, json
import torch

def save_geofno_checkpoint(
    model,
    model_iphi,
    folder: str,
    name: str,
    extra_meta: dict | None = None,
):
    os.makedirs(folder, exist_ok=True)
    model_path = os.path.join(folder, f"{name}_fno.pt")
    iphi_path  = os.path.join(folder, f"{name}_iphi.pt")
    meta_path  = os.path.join(folder, f"{name}_meta.json")

    torch.save(model.state_dict(), model_path)
    torch.save(model_iphi.state_dict(), iphi_path)

    meta = {} if extra_meta is None else dict(extra_meta)
    meta["model_path"] = model_path
    meta["iphi_path"] = iphi_path
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    print("Saved:", model_path)
    print("Saved:", iphi_path)
    print("Saved:", meta_path)

save_geofno_checkpoint(
    model,
    model_iphi,
    folder="/scratch/mnhagen/models/geofno",
    name="elasticity_sigma_geofno",
    extra_meta={
        "dataset": "elasticity",
        "input": "geometry (x,y)",
        "output": "sigma",
        "split": "1600/400 random",
    },
)


Saved: /scratch/mnhagen/models/geofno/elasticity_sigma_geofno_fno.pt
Saved: /scratch/mnhagen/models/geofno/elasticity_sigma_geofno_iphi.pt
Saved: /scratch/mnhagen/models/geofno/elasticity_sigma_geofno_meta.json
